# M1.2 · A 股交易机制

> 策略必须在真实的交易规则内运行。理解交易时段、竞价方式、涨跌停、T+1、撮合规则，才能设计可执行、可成交的策略。
> 本节系统梳理 A 股交易机制，并计算关键规则参数。

## 一、交易时段

| 时段 | 时间 | 方式 |
|---|---|---|
| 开盘集合竞价 | 9:15–9:25 | 集中撮合产生开盘价 |
| 上午连续竞价 | 9:30–11:30 | 逐笔连续撮合 |
| 下午连续竞价 | 13:00–14:57 | 逐笔连续撮合 |
| 收盘集合竞价 | 14:57–15:00 | 集中撮合产生收盘价 |

- 9:20–9:25 不可撤单；午休不交易；
- 盘口和成交在不同时段形成机制不同。

## 二、竞价方式

- **集合竞价**：一段时间内收集的订单一次性集中撮合，按"最大成交量"原则确定一个成交价；
- **连续竞价**：订单到达即与已有订单逐笔撮合，价格优先、时间优先；
- 开盘价、收盘价由集合竞价产生，盘中由连续竞价产生。

## 三、涨跌停与报价规则

In [1]:
# 涨跌停价计算（主板10%，创业板/科创板20%，ST 5%）
def price_limit(prev_close, pct, tick=0.01):
    up = round(prev_close * (1 + pct) / tick) * tick
    down = round(prev_close * (1 - pct) / tick) * tick
    return round(up, 2), round(down, 2)
for board, pct in [('主板', 0.10), ('创业板/科创板', 0.20), ('ST', 0.05)]:
    up, down = price_limit(10.00, pct)
    print('%-10s 涨停 %.2f  跌停 %.2f' % (board, up, down))

主板         涨停 11.00  跌停 9.00
创业板/科创板    涨停 12.00  跌停 8.00
ST         涨停 10.50  跌停 9.50


- 涨跌停板：价格不能超过昨收 ± 限制幅度；涨停时卖单稀缺、跌停时买单稀缺，常无法成交；
- 最小价格变动单位 0.01 元；
- 买入申报数量为 100 股（1 手）的整数倍，卖出余额不足 100 股可零股卖出。

In [2]:
# 手数与最小变动演示
shares = 1250
print('申报 %d 股 -> %d 手余 %d 股（买入须为100整数倍）' %
      (shares, shares//100, shares % 100))
print('价格 10.005 按最小变动单位取整 ->', round(10.005/0.01)*0.01)

申报 1250 股 -> 12 手余 50 股（买入须为100整数倍）
价格 10.005 按最小变动单位取整 -> 10.01


## 四、T+1 交收制度

- A 股股票实行 T+1：当日买入的股票，下一交易日才能卖出；
- 资金方面，当日卖出股票所得资金可当日再买入（资金可用），但取现 T+1；
- 影响：日内"当天买当天卖"不可行，日内策略只能用已有底仓或做 T，回测要遵守 T+1 约束。

## 五、撮合规则：价格优先、时间优先

In [3]:
# 撮合优先级演示：同价位按时间，不同价位按价格
orders = [('BUY', 10.00, 100, '09:30:01'), ('BUY', 10.01, 100, '09:30:02'),
          ('BUY', 10.00, 100, '09:30:03')]
# 买方出价高者优先；同价先到先得
ranked = sorted(orders, key=lambda o: (-o[1], o[3]))
for o in ranked:
    print(o)

('BUY', 10.01, 100, '09:30:02')
('BUY', 10.0, 100, '09:30:01')
('BUY', 10.0, 100, '09:30:03')


- **价格优先**：买入高价、卖出低价优先成交；
- **时间优先**：同价位先申报者优先；
- 限价单排队等待，市价单优先即时成交。

## 六、交易费用（与回测成本呼应）

- 佣金（券商，通常万分之几，有最低收费）；
- 印花税（卖出单边收取）；
- 过户费（沪市）；
- 一次完整买卖的显性成本约万分之几到千分之一，回测要计入（L1 已实测约 10.2bps）。

## 七、这些机制对策略的影响

- 涨跌停：信号可能无法成交，回测要建模涨跌停可成交性；
- T+1：限制日内反转，不能当日止损卖出当日买入；
- 集合竞价：开盘/收盘成交的冲击和价格形成特殊；
- 手数/最小变动：影响拆单和小资金成交；
- 停牌：无法交易，需在回测中处理。

## 八、小结与面试要点

- 交易时段含集合竞价与连续竞价；涨跌停、T+1、100 股整数倍、0.01 最小变动是核心规则。
- 撮合遵循价格优先、时间优先；机制约束必须在回测和实盘中显式建模。

**面试高频：**
- 集合竞价和连续竞价的区别？（答：前者集中一次定价，后者逐笔即时撮合）
- 什么是 T+1，对策略有什么影响？（答：当日买次日才能卖，限制日内交易）
- 涨跌停板会带来什么问题？（答：流动性枯竭、无法成交，回测要考虑）
- 撮合的优先级规则？（答：价格优先、时间优先）

## 动手练习
1. 给定昨收价，计算不同板块的涨跌停价；
2. 列出一个日内策略在 T+1 下可行的替代方案；
3. 说明回测中如何处理涨停日的买入信号。

---
下一节 **M1.3 订单簿与撮合**。